# LeetCode 3894 — Traffic Signal Color bằng Python

Link đề: https://leetcode.com/problems/traffic-signal-color/ · Độ khó: **Easy** · Chủ đề: Math, Simulation, Conditional Logic

Skeleton LeetCode:

```python
class Solution:
    def trafficSignal(self, timer: int) -> str:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua leetcode.doocs.org (Easy, không Premium); các khoảng giá trị của từng màu theo trang doocs nên cần đối chiếu lại với đề gốc nếu muốn chắc chắn từng biên.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chuyển số giây còn lại `timer` thành **màu đèn giao thông** theo bảng luật: `0 -> "Green"`, `30 -> "Orange"`, `31..90 -> "Red"`, còn lại `"Invalid"`.
2. Biết viết chuỗi `if / elif / else`, so sánh **chuỗi** (hai biên khác nhau `==` và `<=`), so sánh liên hoàn `30 < timer <= 90`.
3. Biết thiết kế dạng **bảng luật** (rule table), tra bằng `dict`, `bisect` trên các mốc và bảng tra cứu dựng sẵn.
4. Viết 5 cách khác nhau và đối chiếu mọi giá trị `timer` trong `0..1000` cũng như test ngẫu nhiên có `random.seed`.
5. Tránh lỗi biên: nhầm `30` là Red thay vì Orange, dùng `< 90` thay vì `<= 90`, quên `"Invalid"`, sai chính tả/viết hoa của chuỗi trả về.

## 2. Đề bài & ví dụ

Đèn giao thông có đồng hồ đếm ngược `timer` (số giây còn lại). Màu đèn được xác định như sau:

- `timer == 0`: **Green** (xanh);
- `timer == 30`: **Orange** (vàng/cam);
- `30 < timer <= 90`: **Red** (đỏ);
- các trường hợp còn lại (ví dụ `1..29` hoặc `> 90`): **Invalid**.

Trả về chuỗi tên màu (đúng chính tả, viết hoa chữ đầu như trên).

| Ví dụ | timer | Output | Giải thích |
|---|---|---|---|
| 1 | `60` | `"Red"` | `60` nằm trong `31..90` |
| 2 | `5` | `"Invalid"` | `5` không thỏa luật nào |

Constraints: `0 <= timer <= 1000`.

In [1]:
for t in [0, 5, 29, 30, 31, 60, 90, 91, 1000]:
    print(t, "->", "Green" if t == 0 else "Orange" if t == 30 else "Red" if 30 < t <= 90 else "Invalid")

0 -> Green
5 -> Invalid
29 -> Invalid
30 -> Orange
31 -> Red
60 -> Red
90 -> Red
91 -> Invalid
1000 -> Invalid


## 3. Kiến thức nền

### 3.1. `if / elif / else`

Python kiểm tra các điều kiện **từ trên xuống dưới** và chạy nhánh đầu tiên đúng. `else` chạy khi mọi điều kiện đều sai. Thứ tự các nhánh quan trọng khi các điều kiện có thể chồng lên nhau.

In [2]:
x = 30
if x == 0:
    print("a")
elif x == 30:
    print("b")      # chạy nhánh này, các nhánh sau bị bỏ qua
elif x > 10:
    print("c")
else:
    print("d")

b


### 3.2. So sánh `==`, `<`, `<=` và so sánh liên hoàn

`30 < timer <= 90` nghĩa là `30 < timer and timer <= 90`: loại `30` nhưng giữ `90`. Python cho phép viết liên hoàn như toán học.

In [3]:
for t in (30, 31, 90, 91):
    print(t, 30 < t <= 90, 30 <= t <= 90)

30 False True
31 True True
90 True True
91 False False


### 3.3. Biểu thức điều kiện (ternary)

`A if điều_kiện else B` là biểu thức. Có thể lồng nhau: `"x" if c1 else "y" if c2 else "z"`.

In [4]:
t = 60
print("A" if t == 0 else "B" if t == 30 else "C")

C


### 3.4. `dict` làm bảng tra cứu

`d.get(key, default)` trả giá trị ứng với `key` hoặc `default`. Dùng cho các giá trị **rời rạc** như `0 -> Green`, `30 -> Orange`.

In [5]:
d = {0: "Green", 30: "Orange"}
print(d.get(0, "?"), d.get(30, "?"), d.get(5, "?"))

Green Orange ?


### 3.5. `bisect_right` trên các mốc

Cho các mốc tăng dần `[1, 30, 31, 91]`, `bisect_right(mốc, x)` cho số mốc `<= x`, tức chỉ số của **khoảng** chứa `x`. Khi đó danh sách nhãn có độ dài `len(mốc) + 1` để ánh xạ khoảng -> màu.

In [6]:
from bisect import bisect_right
cuts = [1, 30, 31, 91]
for x in [0, 1, 29, 30, 31, 90, 91, 1000]:
    print(x, bisect_right(cuts, x))

0 0
1 1
29 1
30 2
31 3
90 3
91 4
1000 4


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Chia **trục số** `0..1000` thành các đoạn rời nhau, mỗi đoạn một màu:

| đoạn | `0` | `1..29` | `30` | `31..90` | `91..1000` |
|---|---|---|---|---|---|
| màu | Green | Invalid | Orange | Red | Invalid |

Bài chỉ là **tra bảng**: tìm đoạn chứa `timer`. Cẩn thận ở các **biên**: `0`, `1`, `29`, `30`, `31`, `90`, `91`.

Tính tay: `timer = 30` -> đoạn `30` -> `Orange` (chứ không phải Red, vì Red bắt đầu từ `31`). `timer = 90` -> `Red` (biên phải **bao gồm**). `timer = 91` -> `Invalid`. `timer = 0` -> `Green`.

In [7]:
segments = [(0, 0, "Green"), (1, 29, "Invalid"), (30, 30, "Orange"), (31, 90, "Red"), (91, 1000, "Invalid")]
for lo, hi, name in segments:
    print(f"[{lo:4d}, {hi:4d}] -> {name}")
assert sum(hi - lo + 1 for lo, hi, _ in segments) == 1001     # phủ kín 0..1000, không chồng lấn

[   0,    0] -> Green
[   1,   29] -> Invalid
[  30,   30] -> Orange
[  31,   90] -> Red
[  91, 1000] -> Invalid


## 5. Cách 1 — Brute Force / baseline

Chuỗi `if / elif / else` viết đúng từng luật của đề, theo thứ tự `0`, `30`, `31..90`, còn lại `Invalid`. Đây là cách trực tiếp nhất và cũng đã tối ưu (`O(1)`).

"Nút thắt": nếu luật thay đổi (thêm màu, đổi mốc) ta phải sửa nhiều nhánh `elif`, dễ nhầm biên; các cách sau tách **dữ liệu** (bảng luật) khỏi **thuật toán** tra cứu.

In [8]:
class SolutionBrute:
    def trafficSignal(self, timer):
        if timer == 0:
            return "Green"
        elif timer == 30:
            return "Orange"
        elif 30 < timer <= 90:
            return "Red"
        else:
            return "Invalid"

print(SolutionBrute().trafficSignal(60), SolutionBrute().trafficSignal(5), SolutionBrute().trafficSignal(30))

Red Invalid Orange


**Độ phức tạp:** Time `O(1)`, Space `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** đây là bài **phân loại theo đoạn**. Hai loại đoạn:

- đoạn **một điểm** (`0`, `30`) -> `dict` tra thẳng;
- đoạn **liên tục** (`31..90`) -> kiểm tra bằng so sánh liên hoàn `lo <= timer <= hi`.

Quy tắc kiểm tra thứ tự: các đoạn **không chồng nhau** nên thứ tự kiểm tra không ảnh hưởng nếu ta viết đúng biên. Hãy kiểm chứng bằng cách chạy mọi `timer` từ `0` đến `1000` (chỉ `1001` giá trị).

**Invariant** của cách "bảng luật": các đoạn trong bảng đôi một rời nhau và phủ `0..1000` (tổng độ dài `1001`); khi đó mỗi `timer` thuộc **đúng một** đoạn và trả đúng một màu.

## 7. Các cách tiếp theo

### Cách 2 — `dict` cho điểm đơn, so sánh cho đoạn

`SPECIAL = {0: "Green", 30: "Orange"}`; nếu `timer` có trong `SPECIAL` trả luôn; ngược lại kiểm tra `30 < timer <= 90`.

In [9]:
class SolutionDict:
    SPECIAL = {0: "Green", 30: "Orange"}

    def trafficSignal(self, timer):
        if timer in self.SPECIAL:
            return self.SPECIAL[timer]
        if 30 < timer <= 90:
            return "Red"
        return "Invalid"

print(SolutionDict().trafficSignal(60), SolutionDict().trafficSignal(5), SolutionDict().trafficSignal(30))

Red Invalid Orange


**Độ phức tạp:** Time `O(1)`, Space `O(1)`.

### Cách 3 — Bảng luật `(lo, hi, màu)` và quét tuyến tính

Mỗi luật là một bộ ba; quét bảng, luật đầu tiên có `lo <= timer <= hi` thì trả màu. Không khớp luật nào -> `Invalid`. Thêm/sửa luật chỉ cần sửa bảng.

In [10]:
class SolutionRuleTable:
    RULES = [(0, 0, "Green"), (30, 30, "Orange"), (31, 90, "Red")]

    def trafficSignal(self, timer):
        for lo, hi, color in self.RULES:
            if lo <= timer <= hi:
                return color
        return "Invalid"

print(SolutionRuleTable().trafficSignal(60), SolutionRuleTable().trafficSignal(5), SolutionRuleTable().trafficSignal(30))

Red Invalid Orange


**Độ phức tạp:** Time `O(R)` với `R = 3` luật (hằng số), Space `O(R)`.

### Cách 4 — `bisect_right` trên các mốc

Các mốc `[1, 30, 31, 91]` chia trục số thành 5 đoạn; `labels = [Green, Invalid, Orange, Red, Invalid]`. Nhãn của `timer` là `labels[bisect_right(cuts, timer)]`. Mẫu này hữu ích khi có **rất nhiều** đoạn (tra cứu `O(log R)`).

In [11]:
from bisect import bisect_right

class SolutionBisect:
    CUTS = [1, 30, 31, 91]
    LABELS = ["Green", "Invalid", "Orange", "Red", "Invalid"]

    def trafficSignal(self, timer):
        return self.LABELS[bisect_right(self.CUTS, timer)]

print(SolutionBisect().trafficSignal(60), SolutionBisect().trafficSignal(5), SolutionBisect().trafficSignal(30))

Red Invalid Orange


**Độ phức tạp:** Time `O(log R)`, Space `O(R)`.

### Cách 5 — Dựng sẵn bảng tra cứu cho `0..1000`

Vì `timer` chỉ có `1001` giá trị, ta dựng một lần list `TABLE[t]` rồi trả `TABLE[timer]`. Mỗi truy vấn `O(1)` thật sự; chi phí dựng `O(1001)` chỉ trả một lần (hữu ích khi có hàng triệu truy vấn).

In [12]:
class SolutionTable:
    TABLE = ["Green" if t == 0 else "Orange" if t == 30 else "Red" if 31 <= t <= 90 else "Invalid" for t in range(1001)]

    def trafficSignal(self, timer):
        return self.TABLE[timer]

print(SolutionTable().trafficSignal(60), SolutionTable().trafficSignal(5), SolutionTable().trafficSignal(30), len(SolutionTable.TABLE))

Red Invalid Orange 1001


**Độ phức tạp:** Time `O(1)` mỗi truy vấn (dựng bảng `O(1001)` một lần), Space `O(1001)`.

## 8. TRACE từng bước

Trace Cách 1 cho các giá trị ở biên:

| timer | `== 0`? | `== 30`? | `30 < t <= 90`? | kết quả |
|---|---|---|---|---|
| 0 | có | | | Green |
| 5 | không | không | không | Invalid |
| 30 | không | có | | Orange |
| 31 | không | không | có | Red |
| 90 | không | không | có | Red |
| 91 | không | không | không (91 > 90) | Invalid |

In [13]:
def trace(timer):
    c0 = timer == 0
    c30 = timer == 30
    cred = 30 < timer <= 90
    ans = "Green" if c0 else "Orange" if c30 else "Red" if cred else "Invalid"
    print(f"timer={timer:4d}: ==0:{c0!s:5} ==30:{c30!s:5} 30<t<=90:{cred!s:5} -> {ans}")
    return ans

expected = {0: "Green", 5: "Invalid", 30: "Orange", 31: "Red", 90: "Red", 91: "Invalid", 1000: "Invalid"}
for t, e in expected.items():
    assert trace(t) == e

timer=   0: ==0:True  ==30:False 30<t<=90:False -> Green
timer=   5: ==0:False ==30:False 30<t<=90:False -> Invalid
timer=  30: ==0:False ==30:True  30<t<=90:False -> Orange
timer=  31: ==0:False ==30:False 30<t<=90:True  -> Red
timer=  90: ==0:False ==30:False 30<t<=90:True  -> Red
timer=  91: ==0:False ==30:False 30<t<=90:False -> Invalid
timer=1000: ==0:False ==30:False 30<t<=90:False -> Invalid


In [14]:
def trace_bisect(timer):
    cuts = [1, 30, 31, 91]
    labels = ["Green", "Invalid", "Orange", "Red", "Invalid"]
    k = bisect_right(cuts, timer)
    print(f"timer={timer:4d}: số mốc <= timer = {k} -> labels[{k}] = {labels[k]}")
    return labels[k]

for t in (0, 29, 30, 31, 90, 91):
    trace_bisect(t)

timer=   0: số mốc <= timer = 0 -> labels[0] = Green
timer=  29: số mốc <= timer = 1 -> labels[1] = Invalid
timer=  30: số mốc <= timer = 2 -> labels[2] = Orange
timer=  31: số mốc <= timer = 3 -> labels[3] = Red
timer=  90: số mốc <= timer = 3 -> labels[3] = Red
timer=  91: số mốc <= timer = 4 -> labels[4] = Invalid


## 9. Code nộp LeetCode

Dùng Cách 1 (chuỗi `if / elif / else`): ngắn, dễ đọc, và đã `O(1)`.

In [15]:
from typing import List, Optional


class Solution:
    def trafficSignal(self, timer: int) -> str:
        if timer == 0:
            return "Green"
        if timer == 30:
            return "Orange"
        if 30 < timer <= 90:
            return "Red"
        return "Invalid"


print(Solution().trafficSignal(60), Solution().trafficSignal(5))

Red Invalid


## 10. Vì sao thuật toán đúng

1. **Các đoạn rời nhau và phủ kín.** `{0}`, `{30}`, `[31, 90]` đôi một không giao nhau; phần còn lại của `[0, 1000]` thuộc "Invalid". Nên mỗi `timer` thuộc đúng một loại.
2. **Thứ tự kiểm tra an toàn.** `timer == 0`, `timer == 30` và `30 < timer <= 90` loại trừ lẫn nhau, nên bất kể thứ tự, kết quả như nhau. Điều kiện `30 < timer` (chặt) đảm bảo `30` không rơi vào Red.
3. **Biên phải bao gồm.** `<= 90` giữ `90` trong Red; `91` rơi xuống `Invalid`.
4. **Không có nhánh nào bị bỏ sót.** `return "Invalid"` ở cuối bao trùm mọi giá trị không thỏa luật nào, ví dụ `1..29` và `91..1000`.
5. Các cách khác (bảng luật, `bisect`, bảng dựng sẵn) biểu diễn **cùng một phân hoạch**; việc so sánh với cách 1 trên cả `1001` giá trị ở mục 12 xác nhận điều đó.

## 11. Phân tích Time / Space Complexity

- Cách 1, 2: `O(1)` thời gian, `O(1)` bộ nhớ.
- Cách 3: `O(R)` với `R` là số luật (`R = 3`): hằng số.
- Cách 4: `O(log R)` thời gian, `O(R)` bộ nhớ.
- Cách 5: `O(1)` mỗi truy vấn, `O(1001)` bộ nhớ cho bảng dựng sẵn.

Không có sự khác biệt best / average / worst; chỉ cách 3 phụ thuộc vị trí luật khớp (luật đầu khớp sớm hơn), vẫn là hằng số.

## 12. Bộ test

Vì miền giá trị chỉ có `1001` phần tử, ta kiểm tra **toàn bộ** `timer` từ `0` đến `1000` cho cả 5 cách + code nộp, cộng test ngẫu nhiên có seed và các ví dụ đề.

In [16]:
import random

solutions = [SolutionBrute(), SolutionDict(), SolutionRuleTable(), SolutionBisect(), SolutionTable(), Solution()]

def reference(t):
    if t == 0: return "Green"
    if t == 30: return "Orange"
    if 31 <= t <= 90: return "Red"
    return "Invalid"

# ví dụ của đề và các biên
fixed = {60: "Red", 5: "Invalid", 0: "Green", 30: "Orange", 31: "Red", 90: "Red", 91: "Invalid", 1: "Invalid", 29: "Invalid", 1000: "Invalid"}
for t, e in fixed.items():
    for s in solutions:
        assert s.trafficSignal(t) == e, (type(s).__name__, t)

# toàn bộ miền 0..1000
for t in range(1001):
    for s in solutions:
        assert s.trafficSignal(t) == reference(t), (type(s).__name__, t)

random.seed(38940)
for _ in range(300):
    t = random.randint(0, 1000)
    assert len({s.trafficSignal(t) for s in solutions}) == 1
from collections import Counter
dist = Counter(Solution().trafficSignal(t) for t in range(1001))
print(dist)
assert dist == {"Green": 1, "Orange": 1, "Red": 60, "Invalid": 939}
print("tất cả test OK")

Counter({'Invalid': 939, 'Red': 60, 'Green': 1, 'Orange': 1})
tất cả test OK


### Test thuộc tính

Mỗi màu xuất hiện đúng trên đoạn kỳ vọng: `Green` chỉ ở `0`, `Orange` chỉ ở `30`, `Red` ở đúng `60` giá trị liên tiếp `31..90`, và mọi kết quả thuộc tập `{Green, Orange, Red, Invalid}`.

In [17]:
colors = {c: [t for t in range(1001) if Solution().trafficSignal(t) == c] for c in ("Green", "Orange", "Red", "Invalid")}
assert colors["Green"] == [0] and colors["Orange"] == [30]
assert colors["Red"] == list(range(31, 91))
assert len(colors["Invalid"]) == 1001 - 62
assert set(Solution().trafficSignal(t) for t in range(1001)) == {"Green", "Orange", "Red", "Invalid"}
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: `30` bị tính là Red

Viết `30 <= timer <= 90` hoặc đặt nhánh Red trước nhánh Orange khiến `30` rơi vào Red.

In [18]:
def wrong_red_first(timer):
    if 30 <= timer <= 90:                                          # SAI: gồm cả 30
        return "Red"
    if timer == 0:
        return "Green"
    if timer == 30:
        return "Orange"
    return "Invalid"

print(wrong_red_first(30), "!=", Solution().trafficSignal(30))
assert wrong_red_first(30) == "Red" and Solution().trafficSignal(30) == "Orange"

Red != Orange


### Lỗi 2: dùng `< 90` thay vì `<= 90`

Biên phải của Red **bao gồm** `90`.

In [19]:
def wrong_strict_90(timer):
    if timer == 0:
        return "Green"
    if timer == 30:
        return "Orange"
    if 30 < timer < 90:                                            # SAI: phải <= 90
        return "Red"
    return "Invalid"

print(wrong_strict_90(90), "!=", Solution().trafficSignal(90))
assert wrong_strict_90(90) == "Invalid" and Solution().trafficSignal(90) == "Red"

Invalid != Red


### Lỗi 3: quên `Invalid` — coi mọi `timer > 30` là Red

Bỏ cận trên `<= 90` khiến `91..1000` bị coi là Red.

In [20]:
def wrong_no_upper(timer):
    if timer == 0:
        return "Green"
    if timer == 30:
        return "Orange"
    if timer > 30:                                                 # SAI: thiếu cận trên 90
        return "Red"
    return "Invalid"

print(wrong_no_upper(500), "!=", Solution().trafficSignal(500))
assert wrong_no_upper(500) == "Red" and Solution().trafficSignal(500) == "Invalid"

Red != Invalid


### Lỗi 4: sai chính tả hoặc chữ hoa/thường của chuỗi trả về

LeetCode so sánh chuỗi **chính xác từng ký tự**: `"red"`, `"RED"`, `"Orange "` (có dấu cách thừa) đều sai.

In [21]:
def wrong_case(timer):
    return {"Green": "green", "Orange": "orange", "Red": "red", "Invalid": "invalid"}[Solution().trafficSignal(timer)]   # SAI: chữ thường

print(wrong_case(60), "!=", Solution().trafficSignal(60))
assert wrong_case(60) == "red" and wrong_case(60) != Solution().trafficSignal(60)
assert Solution().trafficSignal(60) == "Red" and Solution().trafficSignal(5) == "Invalid"

red != Red


### Lỗi 5: dùng `or`/`and` sai (điều kiện luôn đúng)

`timer == 0 or 30` luôn truthy vì `30` là số khác 0: Python đọc thành `(timer == 0) or 30`.

In [22]:
def wrong_or(timer):
    if timer == 0 or 30:                                           # SAI: (timer == 0) or 30 -> luôn đúng
        return "Green"
    return "Invalid"

print(wrong_or(60), "!=", Solution().trafficSignal(60))
assert wrong_or(60) == "Green" and Solution().trafficSignal(60) == "Red"
assert (60 == 0 or 30) == 30 and (60 in (0, 30)) is False

Green != Red


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề dạng "nếu ... thì màu/nhãn ..." -> **phân hoạch trục số** thành đoạn, vẽ bảng đoạn trước khi code.
- Chú ý **loại biên**: `<` hay `<=`; ghi rõ biên trái và biên phải của từng đoạn.
- Khi miền giá trị nhỏ (`<= 1000`), có thể **kiểm tra toàn bộ** để chắc chắn không sai biên.
- Dùng `in (a, b)` thay `== a or == b`; tránh `x == a or b`.
- Chuỗi trả về phải **khớp từng ký tự** với đề: sao chép nguyên văn.

## 15. Biến thể của bài

1. **FizzBuzz** (412): phân loại theo bội số.
2. **Water Bottles / Calculate Money in Leetcode Bank**-style: các bài mô phỏng cơ bản.
3. **Categorize Box According to Criteria** (2525): phân loại theo điều kiện, trả về nhãn chuỗi.
4. **Find the Winner / Tic-Tac-Toe**: mô phỏng với các nhánh điều kiện.
5. Biến thể: thêm màu Yellow ở `1..5` hoặc chu kỳ lặp (đèn đổi màu theo `timer % 120`).
6. Biến thể: dựng **hàm ngược**: cho màu, liệt kê mọi `timer` có thể.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `if / elif` | viết đúng từng luật | `O(1)` | `O(1)` | Nộp bài |
| 2. `dict` + so sánh | điểm đơn tra bảng, đoạn so sánh | `O(1)` | `O(1)` | Có nhiều điểm đơn |
| 3. Bảng luật | `(lo, hi, màu)` quét tuyến tính | `O(R)` | `O(R)` | Luật hay thay đổi |
| 4. `bisect` | mốc + nhãn | `O(log R)` | `O(R)` | Rất nhiều đoạn |
| 5. Bảng dựng sẵn | `TABLE[timer]` | `O(1)` | `O(1001)` | Hàng triệu truy vấn |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().trafficSignal(31)`, `Solution().trafficSignal(100)` và `Solution().trafficSignal(0)` ra gì?

**Bài 2 (tính tay):** có bao nhiêu giá trị `timer` trong `0..1000` cho kết quả `Invalid`?

**Bài 3 (điền TODO):** hoàn thành `grade(score)`: `score >= 90` -> `"A"`, `80..89` -> `"B"`, `70..79` -> `"C"`, ngược lại `"F"`.

**Bài 4 (tự viết lại):** viết `signal_with_cycle(timer)`: nếu `timer >= 120` thì thay `timer` bằng `timer % 120` rồi áp dụng luật gốc.

**Bài 5 (mini challenge):** viết `timers_for(color)` trả về danh sách mọi `timer` trong `0..1000` cho ra `color` (dùng `Solution`), và kiểm tra các độ dài.

In [23]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().trafficSignal(31), Solution().trafficSignal(100), Solution().trafficSignal(0))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def grade_todo(score):
    # TODO: trả "A" / "B" / "C" / "F"
    return "F"

Red Invalid Green


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `31` nằm trong `31..90`; `100 > 90`; `0` là biên đặc biệt. Bài 2: tổng giá trị `1001` trừ các giá trị Green, Orange, Red.

**Mức 2 (vừa):** Bài 3: kiểm tra từ điểm cao xuống thấp bằng `elif`. Bài 4: `% 120` chỉ áp dụng khi `timer >= 120`; với `timer < 120` giữ nguyên. Bài 5: `[t for t in range(1001) if ...]`.

**Mức 3 (gần đáp án):** Bài 1: `Red`, `Invalid`, `Green`. Bài 2: `1001 - 1 - 1 - 60 = 939`. Bài 4: `signal_with_cycle(150)` -> `150 % 120 = 30` -> `Orange`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().trafficSignal(31) == "Red" and Solution().trafficSignal(100) == "Invalid" and Solution().trafficSignal(0) == "Green"
assert sum(1 for t in range(1001) if Solution().trafficSignal(t) == "Invalid") == 939

# Bài 3
def grade(score):
    if score >= 90:
        return "A"
    elif score >= 80:
        return "B"
    elif score >= 70:
        return "C"
    return "F"
assert [grade(s) for s in (95, 90, 89, 80, 79, 70, 69, 0)] == ["A", "A", "B", "B", "C", "C", "F", "F"]

# Bài 4
def signal_with_cycle(timer):
    if timer >= 120:
        timer %= 120
    return Solution().trafficSignal(timer)
assert signal_with_cycle(150) == "Orange" and signal_with_cycle(120) == "Green" and signal_with_cycle(60) == "Red"
assert signal_with_cycle(125) == "Invalid" and signal_with_cycle(1000) == Solution().trafficSignal(1000 % 120)

# Bài 5
def timers_for(color):
    return [t for t in range(1001) if Solution().trafficSignal(t) == color]
assert timers_for("Green") == [0] and timers_for("Orange") == [30]
assert len(timers_for("Red")) == 60 and timers_for("Red")[0] == 31 and timers_for("Red")[-1] == 90
assert timers_for("Blue") == []
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "phân loại theo đoạn":**

```python
if timer == 0:        return "Green"
if timer == 30:       return "Orange"
if 30 < timer <= 90:  return "Red"
return "Invalid"
```

**Checklist trước khi submit:**

- [ ] Đã vẽ bảng đoạn và kiểm tra các biên `0, 1, 29, 30, 31, 90, 91`?
- [ ] `30` là Orange (không phải Red); `90` là Red (biên phải bao gồm)?
- [ ] Có nhánh cuối `Invalid` cho mọi giá trị còn lại?
- [ ] Chuỗi trả về khớp chính xác: `Green`, `Orange`, `Red`, `Invalid`?
- [ ] Không viết `timer == 0 or 30` (luôn đúng)?
- [ ] Chữ ký đúng `trafficSignal(self, timer: int) -> str`?